# block-group-stack — worked example 2: Push a tensor through a BlockGroup and check spatial halving

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `block-group-stack`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Because only block 0 of a BlockGroup carries `first_stride`, the whole group downsamples the spatial dimensions exactly once (by `first_stride`) and changes channels exactly once. All later blocks are shape-preserving, so the group's output shape is fully determined by block 0's stride and `out_feats`.

## Worked solution

We confirm the shape behaviour empirically.

**Step 1 - build the group.** Block 0 is `ResBlock(in_feats, out_feats, first_stride)`; the rest are `(out_feats, out_feats, 1)`. With a 1x1 conv of stride `s`, a spatial size `H` becomes `ceil(H/s)` (here `H//s` for even sizes).

**Step 2 - make an input.** We seed with `t.manual_seed(0)` then draw `x` of shape `(batch, in_feats, H, W)`. Re-seeding inside keeps the draw reproducible regardless of outer seeding.

**Step 3 - forward pass.** `nn.Sequential` chains the blocks, so `group(x)` runs block 0 (channels `in_feats->out_feats`, stride `s`) then the identity-shaped blocks. The channel dim becomes `out_feats` and the spatial dims are divided by `s` once - the later stride-1 blocks leave them alone.

**Step 4 - assert.** Output channels equal `out_feats`; output H,W equal `H//s, W//s`. Printing the in/out shapes makes the single-downsample invariant visible.

In [ ]:
import torch as t
import torch.nn as nn

class ResBlock(nn.Module):
    def __init__(self, in_feats, out_feats, first_stride=1):
        super().__init__()
        self.in_feats = in_feats
        self.out_feats = out_feats
        self.first_stride = first_stride
        self.proj = nn.Conv2d(in_feats, out_feats, kernel_size=1, stride=first_stride)
    def forward(self, x):
        return self.proj(x)

def make_block_group(in_feats, out_feats, n_blocks, first_stride):
    blocks = [ResBlock(in_feats, out_feats, first_stride=first_stride)]
    for _ in range(n_blocks - 1):
        blocks.append(ResBlock(out_feats, out_feats, first_stride=1))
    return nn.Sequential(*blocks)

t.manual_seed(0)
group = make_block_group(in_feats=8, out_feats=24, n_blocks=3, first_stride=2)
x = t.randn(2, 8, 16, 16)
y = group(x)
print('input shape :', tuple(x.shape))
print('output shape:', tuple(y.shape))
assert y.shape == (2, 24, 8, 8)
print('downsampled once, channels 8 -> 24: OK')